3.2:DATA PREPARATION

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [4]:
# data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'



In [5]:
# !wget $data -O data-week-3.csv

In [2]:
df=pd.read_csv('data-week-3.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [4]:
# making everything uniform
df.columns= df.columns.str.lower().str.replace(' ', '_')
categorical_columns= list(df.dtypes[df.dtypes=='str'].index)


for c in categorical_columns:
    df[c]=df[c].str.lower().str.replace(' ', '_')

In [5]:
# transpose the dataframe to see the columns better
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [6]:
tc=pd.to_numeric(df.totalcharges, errors='coerce')


In [7]:
tc.isnull().sum()

np.int64(11)

In [8]:
df[tc.isnull()][['customerid','totalcharges']]

,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


In [10]:
# These are customers who just joined - their tenure is small and they haven't been billed yet, so their total charges are not available.
#  For them we simply fill the missing values with zero:

df.totalcharges= pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges= df.totalcharges.fillna(0)
df.totalcharges

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: totalcharges, Length: 7043, dtype: float64

In [11]:
# encoding the target variable
df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: str

In [12]:
(df.churn == 'yes').head()

0    False
1    False
2     True
3    False
4     True
Name: churn, dtype: bool

In [13]:
# |for classification we need numbers, hence why we are casting the churn column to int. 1 for yes and 0 for no.
df.churn=(df.churn == 'yes').astype(int)


In [14]:
df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

3.3 : SETTING UP THE VALIDATION FRAMEWORK

In [15]:
# splitting the data 
from sklearn.model_selection import train_test_split

In [16]:
df_full_train,df_test= train_test_split(df,test_size=0.2,random_state=1)

In [17]:
# 20% of 80 is 25% hence using test_size as 0.25
df_train, df_val = train_test_split(df_full_train, test_size=0.25,random_state=1)

In [18]:
len(df_train),len(df_val), len(df_test)

(4225, 1409, 1409)

In [19]:
# resetting the index
df_train= df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

In [20]:
# ISOLATING THE TARGET
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values


In [21]:
# deleting the column from the dataframe
del df_train['churn']
del df_val['churn']
del df_test['churn']

3.4: EXPLORATORY DATA ANALYSIS(EDA)

Now we explore the training data: we check for missing values, look at the distribution of the target variable, and go through the numerical and categorical columns. 
For EDA we use the full train set - train and validation together - because we will also use it to compute feature importance later.

In [22]:
df_full_train = df_full_train.reset_index(drop=True)

In [23]:
df_full_train.isnull().sum()  

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [24]:
# checking the churn rate, how many customers churned and how many remained
df_full_train.churn.value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [25]:
# since the churn is a column of ones and zeroes this is a shortcut of calculating the churn rate
df_full_train.churn.mean()

np.float64(0.26996805111821087)

In [26]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

In [27]:
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [28]:
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

3.5: FEATURE IMPORTANCE: Churn rate and risk ratio

In [29]:
# churn rate per group
global_churn = df_full_train.churn.mean()
global_churn

np.float64(0.26996805111821087)

In [30]:
# Now we look at individual variables. Let's start with gender:
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()


In [31]:
global_churn - churn_female

np.float64(-0.006855983216553063)

In [32]:
global_churn - churn_male

np.float64(0.006754520462819769)

In [33]:
# Both differences are close to zero, so gender is probably not an important feature: knowing whether a customer is a man or a woman tells us almost nothing about churn.
# Now let's look at a more promising variable - whether the customer has a partner. First, how many customers are there in each group?

df_full_train.partner.value_counts()

partner
no     2932
yes    2702
Name: count, dtype: int64

In [34]:
# churn ratios
churn_partner = df_full_train[df_full_train.partner== 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner== 'no'].churn.mean()

In [35]:
churn_partner

np.float64(0.20503330866025166)

In [36]:
churn_no_partner

np.float64(0.3298090040927694)

In [37]:
global_churn - churn_partner

np.float64(0.06493474245795922)

In [38]:
global_churn - churn_no_partner 

np.float64(-0.05984095297455855)

In [39]:
churn_no_partner / global_churn

np.float64(1.2216593879412643)

In [40]:
from IPython.display import display

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


3.6 FEATURE IMPORTANCE: MUTUAL INFORMATION
measure of mutual dependence between two variables

In [42]:
from sklearn.metrics import mutual_info_score

In [43]:
mutual_info_score(df_full_train.churn,df_full_train.contract)

0.0983203874041556

In [44]:
mutual_info_score(df_full_train.churn, df_full_train.partner)

0.009967689095399745

In [46]:
mutual_info_score(df_full_train.gender,df_full_train.churn)

0.0001174846211139946

In [48]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

In [50]:
mi=df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)


contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

3.7  FEATURE IMPORTANCE: CORRELATION

In [51]:
df_full_train.tenure.max()

np.int64(72)

In [61]:
df_full_train[numerical].corrwith(df_full_train.churn).abs()

tenure            0.351885
monthlycharges    0.196805
totalcharges      0.196353
dtype: float64

In [ ]:
# perhaps we can look at the churn rate for customers who have been with the company  for 2 months or less:
df_full_train[df_full_train.tenure <=2].churn.mean()

np.float64(0.5953420669577875)

In [ ]:
# perhaps we can look at the churn rate for customers who have been with the company for more than 2 months but less than a year:
df_full_train[(df_full_train.tenure >2) & (df_full_train.tenure <=12)].churn.mean()

np.float64(0.3994413407821229)

In [ ]:
# perhaps we can look at the churn rate for customers who have been with the company for more than a year:
df_full_train[df_full_train.tenure >12].churn.mean()

# in tenure there is a decrease in churn rate hence a negative correlation between tenure and churn.

np.float64(0.17634908339788277)

In [58]:
# monthly charges for less than 20 dollars
df_full_train[df_full_train.monthlycharges <20].churn.mean()

np.float64(0.08829568788501027)

In [59]:
# monthly charges for more than 20 dollars and less than 50 dollars
df_full_train[(df_full_train.monthlycharges >20) & (df_full_train.monthlycharges <=50)].churn.mean()

np.float64(0.18340943683409436)

In [ ]:
# montly charges for more than 50 dollars churn rate
df_full_train[df_full_train.monthlycharges >50].churn.mean()

# in monthly charges there is an increase in churn rate hence a positive  correalation between monthly charges and churn.

np.float64(0.32499341585462205)

3.8 : ONE-HOT ENCODING

In [62]:
from sklearn.feature_extraction import DictVectorizer

In [87]:
df_train[['gender','contract','tenure']].iloc[:10]

,gender,contract,tenure
0,female,two_year,72
1,male,month-to-month,10
2,female,month-to-month,5
3,female,month-to-month,5
4,female,two_year,18
5,male,month-to-month,4
6,male,month-to-month,1
7,female,month-to-month,1
8,female,two_year,72
9,female,month-to-month,6


In [ ]:

train_dicts= df_train[ categorical + numerical].iloc[:100].to_dict(orient='records')

In [94]:
dv= DictVectorizer(sparse=False)

In [95]:
x_train= dv.fit_transform(train_dicts)

In [96]:
val_dicts= df_val[categorical + numerical].to_dict(orient='records')

In [97]:
x_val= dv.transform(val_dicts)